# 03 — Continuum Memory System

Walkthrough of `hope.memory.ContinuumMemorySystem` (CMS), §7.1 of the paper.

Eq. 70: $y_t = \mathrm{MLP}^{(f_k)}(\mathrm{MLP}^{(f_{k-1})}(\cdots\mathrm{MLP}^{(f_1)}(x_t)))$.

Eq. 71: each bank $\ell$ is updated only when $t \equiv 0 \pmod{C^{(\ell)}}$ — its own chunk size.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from hope.memory import ContinuumMemorySystem

tf.random.set_seed(0)
np.random.seed(0)

DIM = 8
N_STEPS = 200
tokens = tf.random.normal((N_STEPS, DIM))
print('seeded', N_STEPS, 'random tokens of dim', DIM)


## Three banks with frequencies (1, 4, 16)

Smaller `update_every` = faster bank. We log $\|M_\ell\|_F$ at every step for every bank.

In [ ]:
cms = ContinuumMemorySystem(
    dim=DIM,
    update_every=(1, 4, 16),
    decays=(0.01, 0.005, 0.001),
    rule='hebbian',
    learning_rate=0.1,
)

per_bank_norms = [[] for _ in cms.banks]
outputs = []
for t in range(N_STEPS):
    y = cms.forward_step(tokens[t], t)
    outputs.append(y.numpy())
    for i, bank in enumerate(cms.banks):
        per_bank_norms[i].append(float(tf.norm(bank.memory.memory)))

outputs = np.stack(outputs)
print('output shape:', outputs.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
labels = [f'bank {i} (update_every={b.update_every}, decay={b.decay})' for i, b in enumerate(cms.banks)]
for series, label in zip(per_bank_norms, labels):
    ax.plot(series, label=label)
ax.set_xlabel('step')
ax.set_ylabel('||M_l||_F')
ax.set_title('CMS per-bank memory norm over time')
ax.legend()
fig.tight_layout()
plt.show()


Faster banks react to every token and grow quickly; slower banks barely move. That asymmetry is what protects long-term knowledge from being overwritten — §7.1's catastrophic-forgetting argument.